# Regression3 — permanent architecture (post-pivot)

This notebook implements the confirmed architecture from `architecture_pivot_instructions.txt` **v2** (2026-07-20, supersedes v1), generalizing `Regression2.ipynb` Iteration 7 into the standing pipeline rather than treating it as one experiment among many. See `PROJECT_NOTES.txt` for full project history.

Fixed decisions in this notebook (not parameters to sweep anymore):
- **Target**: `approved_amount` — resolved after comparing invoice/approved/paid in `Regression2.ipynb` Iterations 5-7.
- **X excludes all four amount fields**: `invoice_amount`, `estimate_amount`, `approved_amount`, `paid_amount`. Audited below, not assumed.
- **Models**: only `GradientBoosting` and `HistGradientBoosting` — these were the best or near-best performers across every iteration in `Regression2.ipynb`; dropped the other 5 for simplicity.
- **part_action features**: `replaced_cost_total`, `repaired_cost_total`, `reused_cost_total`, `num_parts_replaced`, `num_parts_repaired`, `num_parts_reused`, `pct_cost_replaced` — this exact set per v2 section 0. The 5+5 named slot columns (`replaced_part_1..5`/`repaired_part_1..5`) are explicitly NOT features here — they belong to the explain-why layer in `Score_New_Claims.ipynb`.
- **Inference requires live historical data**: `score_new_claim()` takes `historical_df` as a required parameter (v2 section 2) — the shop-average lookup is computed at call time, not frozen into the saved model.
- These are placeholder/synthetic part_action data pending real ClaimCenter fields — do not present results built on them as reflecting real client behavior.

`Regression2.ipynb` is left as-is as the historical experiment log; this notebook is the forward-looking one.

In [151]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load and merge

`Claim_main.csv` (one row per `repair_id`) merged with `repair_parts_action_summary.csv` (also one row per `repair_id`, confirmed unique PK in the README). Inner-merge should not drop any rows if both files share the same `repair_id` set — verified below rather than assumed.

In [152]:
claims = pd.read_csv('Claim_main.csv')
parts_action = pd.read_csv('repair_parts_action_summary.csv')

print("claims rows:", len(claims), "| parts_action rows:", len(parts_action))
print("repair_id sets equal:", set(claims['repair_id']) == set(parts_action['repair_id']))

df = claims.merge(parts_action.drop(columns=['claim_id', 'vehicle_id']), on='repair_id', how='inner', validate='one_to_one')
assert len(df) == len(claims), "merge dropped or duplicated rows"
print("merged rows:", len(df))
df.head()

claims rows: 10241 | parts_action rows: 10241
repair_id sets equal: True
merged rows: 10241


,repair_id,claim_id,vehicle_id,repair_shop_contact_id,invoice_amount,estimate_amount,approved_amount,paid_amount,repair_start_date,repair_end_date,...,num_parts_replaced,replaced_cost_total,overflow_replaced_count,num_parts_repaired,repaired_cost_total,overflow_repaired_count,num_parts_reused,reused_cost_total,total_parts_cost,pct_cost_replaced
0,1,1,1798,209,5113.35,5831.11,4831.38,4871.05,2026-07-08,2026-07-25,...,1,1339.61,0,1,1047.92,0,0,0.00,2387.53,0.5611
1,2,2,3026,12,826.96,1033.44,768.75,746.74,2026-06-21,2026-06-22,...,0,0.00,0,0,0.00,0,1,469.91,469.91,0.0000
2,3,3,6013,88,1091.33,1038.62,964.03,949.65,2025-02-18,2025-02-25,...,0,0.00,0,1,971.74,0,0,0.00,971.74,0.0000
3,4,4,64,164,1754.45,1711.16,1811.61,1789.23,2026-02-20,2026-02-22,...,1,1054.89,0,0,0.00,0,0,0.00,1054.89,1.0000
4,5,5,4714,106,1285.22,1151.50,1104.23,1153.84,2025-03-10,2025-03-18,...,0,0.00,0,0,0.00,0,1,1026.89,1026.89,0.0000


## Feature engineering

Base fields (same as `Regression2.ipynb`) plus the part_action features. Per `architecture_pivot_instructions.txt` v2, the ML-ready part_action feature set is `replaced_cost_total`, `repaired_cost_total`, `reused_cost_total`, `num_parts_replaced`, `num_parts_repaired`, `num_parts_reused`, `pct_cost_replaced` -- note this now includes raw dollar totals, on explicit instruction, overriding the earlier general dollar-leakage exclusion rule for this specific case. The 5+5 named part-slot columns (`replaced_part_1..5`/`repaired_part_1..5`) are NOT features here -- v2 is explicit that they're for the explain-why layer (`find_similar_historical_claims`), not X, so this notebook doesn't touch them at all.

In [153]:
SEVERITY_ORDER = {"Minor": 0, "Moderate": 1, "Severe": 2, "Total Loss": 3}
df["damage_severity_ordinal"] = df["damage_severity"].map(SEVERITY_ORDER)
df["initial_damage_assessment_ordinal"] = df["initial_damage_assessment"].map(SEVERITY_ORDER)
df["airbags_deployed"] = df["airbags_deployed"].astype(int)
df["is_supplemental"] = (df["cost_category"] == "Supplemental").astype(int)
df["has_catastrophe_code"] = df["catastrophe_code"].notna().astype(int)

df["repair_start_date"] = pd.to_datetime(df["repair_start_date"])
df["repair_end_date"] = pd.to_datetime(df["repair_end_date"])
df["repair_duration_days"] = (df["repair_end_date"] - df["repair_start_date"]).dt.days.clip(lower=1)
df["vehicle_age"] = pd.Timestamp.now().year - df["vehicle_year"]

df["is_repeat_vehicle"] = df["is_repeat_vehicle"].astype(int)
df["loss_date"] = pd.to_datetime(df["loss_date"])
df["loss_month"] = df["loss_date"].dt.month

# SYNTHETIC / PLACEHOLDER -- part_action data stands in for a real ClaimCenter field, see README_part_action_data.txt
df["replaced_cost_total"] = df["replaced_cost_total"].fillna(0)
df["repaired_cost_total"] = df["repaired_cost_total"].fillna(0)
df["reused_cost_total"] = df["reused_cost_total"].fillna(0)
df["num_parts_replaced"] = df["num_parts_replaced"].fillna(0)
df["num_parts_repaired"] = df["num_parts_repaired"].fillna(0)
df["num_parts_reused"] = df["num_parts_reused"].fillna(0)
df["pct_cost_replaced"] = df["pct_cost_replaced"].fillna(0)

## Validate remaining schema fields before finalizing the candidate list

`Claim_main.csv` has 8 columns that were carried over unvalidated from `Regression2.ipynb` (never explicitly tested there either): `weather_conditions`, `road_conditions`, `loss_date`, `reported_date`, `cost_type`, `shop_city`, `shop_state`, `is_repeat_vehicle`. Testing each against `approved_amount` (ANOVA for categorical, correlation for numeric/derived) before deciding what belongs in the candidate lists below, per the project's standing validate-before-include rule.

In [154]:
df["reported_date"] = pd.to_datetime(df["reported_date"])
df["reporting_lag_days"] = (df["reported_date"] - df["loss_date"]).dt.days

for col in ["weather_conditions", "road_conditions", "cost_type", "shop_city", "shop_state", "is_repeat_vehicle", "loss_month"]:
    groups = [df.loc[df[col] == v, "approved_amount"] for v in df[col].dropna().unique()]
    f, p = stats.f_oneway(*groups)
    print(f"{col:20s}  ANOVA F={f:8.3f}  p={p:.4g}  n_groups={len(groups)}")

r, p = stats.pearsonr(df["reporting_lag_days"], df["approved_amount"])
print(f"{'reporting_lag_days':20s}  corr r={r:+.3f}  p={p:.4g}")

print()
print("Significant (p<0.05): weather_conditions, is_repeat_vehicle, loss_month -- added to candidate lists below.")
print("Not significant, excluded: road_conditions, cost_type, shop_state, reporting_lag_days.")
print("Borderline (p~0.09), excluded: shop_city -- doesn't clear 0.05, and is already largely captured by the shop_avg encoding.")

weather_conditions    ANOVA F=   4.474  p=0.0004493  n_groups=6
road_conditions       ANOVA F=   0.769  p=0.5452  n_groups=5
cost_type             ANOVA F=   0.144  p=0.9817  n_groups=6
shop_city             ANOVA F=   1.457  p=0.09011  n_groups=20
shop_state            ANOVA F=   1.131  p=0.3361  n_groups=10
is_repeat_vehicle     ANOVA F=  14.104  p=0.000174  n_groups=2
loss_month            ANOVA F=   2.130  p=0.01545  n_groups=12
reporting_lag_days    corr r=-0.001  p=0.9086

Significant (p<0.05): weather_conditions, is_repeat_vehicle, loss_month -- added to candidate lists below.
Not significant, excluded: road_conditions, cost_type, shop_state, reporting_lag_days.
Borderline (p~0.09), excluded: shop_city -- doesn't clear 0.05, and is already largely captured by the shop_avg encoding.


**Update**: `shop_city`/`shop_state` were excluded above based on this ANOVA result, but `architecture_pivot_instructions.txt` later explicitly confirmed they must remain in `X` regardless -- this is the stakeholders' literal ask (vehicle/repair/shop characteristics as the feature set), not a statistical judgment call. Both are included in `CANDIDATE_CATEGORICAL` below. The ANOVA result above is left as-is since it's still an accurate record of what the data alone says.

## Audit: confirm no amount field ever enters X

Per the pivot instructions, verify explicitly rather than assume.

In [155]:
AMOUNT_FIELDS = {"invoice_amount", "estimate_amount", "approved_amount", "paid_amount"}

CANDIDATE_NUMERIC = ["vehicle_age", "liability_percentage", "repair_duration_days", "vehicle_claim_count",
    "damage_severity_ordinal", "initial_damage_assessment_ordinal", "airbags_deployed", "is_supplemental",
    "has_catastrophe_code", "reserve_amount", "is_repeat_vehicle"]

# shop_city/shop_state added per architecture_pivot_instructions.txt's explicit confirmation that they
# must remain in X -- overrides the earlier ANOVA finding (shop_city p=0.09 borderline, shop_state
# p=0.336 not significant) since this is a direct stakeholder requirement, not a statistical call.
CANDIDATE_CATEGORICAL = ["make", "model", "claim_severity", "incident_state", "loss_cause", "point_of_impact",
    "weather_conditions", "loss_month", "shop_city", "shop_state"]

# Per architecture_pivot_instructions.txt v2 section 0: these 7 are THE ML-ready part_action
# features (replaced_cost_total/repaired_cost_total/reused_cost_total ARE included here despite
# being raw dollar totals -- explicit instruction, overriding the earlier leakage-avoidance stance).
# The named slot columns (replaced_part_1..5/repaired_part_1..5) are explicitly NOT model features --
# they're for the explain-why layer (find_similar_historical_claims in Score_New_Claims.ipynb), not X.
PART_ACTION_CANDIDATES = ["replaced_cost_total", "repaired_cost_total", "reused_cost_total",
    "num_parts_replaced", "num_parts_repaired", "num_parts_reused", "pct_cost_replaced"]

leaked = AMOUNT_FIELDS & set(CANDIDATE_NUMERIC + CANDIDATE_CATEGORICAL + PART_ACTION_CANDIDATES)
assert not leaked, f"amount field(s) leaked into candidate features: {leaked}"
print("Audit passed: no amount field present in any candidate feature list.")
print("Note: reserve_amount is NOT one of the 4 banned fields and was not named by stakeholders -- left in X, flagging this back per the pivot instructions rather than deciding silently.")

Audit passed: no amount field present in any candidate feature list.
Note: reserve_amount is NOT one of the 4 banned fields and was not named by stakeholders -- left in X, flagging this back per the pivot instructions rather than deciding silently.


## Train/test split

One split, done once. `approved_amount` (the fixed target) and `invoice_amount` (needed later purely as comparison data for `score_new_claim`, never as a feature) are carried into `train_df`/`test_df` alongside the candidate features.

In [156]:
split_cols = list(dict.fromkeys(CANDIDATE_NUMERIC + CANDIDATE_CATEGORICAL + PART_ACTION_CANDIDATES +
    ["repair_shop_contact_id", "approved_amount", "invoice_amount"]))
train_df, test_df = train_test_split(df[split_cols], test_size=0.2, random_state=42)

## Reusable functions

Same design as `Regression2.ipynb` (X/y/feature lists as parameters), trimmed to the two models that are actually being kept: `GradientBoosting`, `HistGradientBoosting`.

In [157]:
def build_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ])

DEFAULT_PARAMS = {
    "GradientBoosting": {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.1},
    "HistGradientBoosting": {},
}

MODEL_GRIDS = {
    "GradientBoosting": {"model__n_estimators": [100, 200], "model__learning_rate": [0.05, 0.1], "model__max_depth": [2, 3]},
    "HistGradientBoosting": {"model__max_iter": [100, 200], "model__learning_rate": [0.05, 0.1], "model__max_depth": [None, 5, 10]},
}

def make_estimator(name, params):
    if name == "GradientBoosting": return GradientBoostingRegressor(random_state=42, **params)
    elif name == "HistGradientBoosting": return HistGradientBoostingRegressor(random_state=42, **params)

def fit_all_models(X_train, y_train, numeric_features, categorical_features, params=None):
    params = params or DEFAULT_PARAMS
    fitted = {}
    for name in MODEL_GRIDS:
        est = make_estimator(name, params.get(name, {}))
        pipe = Pipeline([("pre", build_preprocessor(numeric_features, categorical_features)), ("model", est)])
        pipe.fit(X_train[numeric_features + categorical_features], y_train)
        fitted[name] = pipe
    return fitted

In [158]:
def add_shop_avg(train_df, test_df, target_col, shop_id_col="repair_shop_contact_id"):
    shop_avg = train_df.groupby(shop_id_col)[target_col].mean()
    global_avg = train_df[target_col].mean()
    col_name = f"shop_avg_{target_col}"
    train_df[col_name] = train_df[shop_id_col].map(shop_avg)
    test_df[col_name] = test_df[shop_id_col].map(shop_avg).fillna(global_avg)
    return col_name

def prepare_target(train_df, test_df, target_col, shop_id_col="repair_shop_contact_id"):
    shop_avg_col = add_shop_avg(train_df, test_df, target_col, shop_id_col)
    return train_df[target_col], test_df[target_col], shop_avg_col

In [159]:
def tune_models(X_train, y_train, numeric_features, categorical_features, n_iter=6, cv=3, random_state=42):
    best_params = {}
    for name, grid in MODEL_GRIDS.items():
        pipe = Pipeline([("pre", build_preprocessor(numeric_features, categorical_features)), ("model", make_estimator(name, {}))])
        n = min(n_iter, int(np.prod([len(v) for v in grid.values()])))
        search = RandomizedSearchCV(pipe, grid, n_iter=n, cv=cv, scoring="neg_mean_absolute_error", random_state=random_state, n_jobs=4)
        search.fit(X_train[numeric_features + categorical_features], y_train)
        best_params[name] = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
    return best_params

In [160]:
def compare_models(fitted_models, X_test, y_test, numeric_features, categorical_features):
    rows = []
    preds = {}
    for name, model in fitted_models.items():
        pred = model.predict(X_test[numeric_features + categorical_features])
        preds[name] = pred
        rows.append({"model": name,
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": mean_squared_error(y_test, pred) ** 0.5,
            "R2": r2_score(y_test, pred)})
    comparison_df = pd.DataFrame(rows).set_index("model")
    return comparison_df, preds

In [161]:
def run_comparison(X_train, X_test, y_train, y_test, numeric_features, categorical_features, params=None):
    fitted = fit_all_models(X_train, y_train, numeric_features, categorical_features, params=params)
    comparison_df, preds = compare_models(fitted, X_test, y_test, numeric_features, categorical_features)
    return comparison_df, fitted, preds

def test_target(train_df, test_df, target_col, numeric_features, categorical_features, params=None):
    y_train, y_test, shop_avg_col = prepare_target(train_df, test_df, target_col)
    features = [c for c in numeric_features if not c.startswith("shop_avg_")] + [shop_avg_col]
    comparison_df, fitted, preds = run_comparison(train_df, test_df, y_train, y_test, features, categorical_features, params=params)
    return comparison_df, fitted, preds

In [162]:
def plot_r2(comparison_df, title=""):
    fig, ax = plt.subplots(figsize=(6, 4))
    comparison_df["R2"].plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1)
    ax.set_ylabel("R2")
    ax.set_title(title)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Fixed target: `approved_amount`

In [163]:
TARGET = "approved_amount"
y_train, y_test, SHOP_AVG_COL = prepare_target(train_df, test_df, TARGET)
X_train, X_test = train_df, test_df

## Statistically validate the new part_action features before adding them to X

Pearson correlation of each candidate part_action feature against `approved_amount`, computed on training data only. Per the project's standing rule, features are only added to X once they show a real, non-trivial relationship with the target -- not on assumption.

In [164]:
for col in PART_ACTION_CANDIDATES:
    r, p = stats.pearsonr(X_train[col], y_train)
    print(f"{col:22s}  r={r:+.3f}  p={p:.4g}")

replaced_cost_total     r=+0.768  p=0
repaired_cost_total     r=+0.123  p=8.403e-29
reused_cost_total       r=+0.044  p=6.256e-05
num_parts_replaced      r=+0.481  p=0
num_parts_repaired      r=-0.182  p=3.842e-62
num_parts_reused        r=-0.086  p=5.175e-15
pct_cost_replaced       r=+0.330  p=1.496e-207


## Baseline — Iteration 7 generalized

`estimate_amount` and all other amount fields excluded from the start (not toggled per-iteration anymore). This is the standing baseline going forward.

In [165]:
numeric_baseline = ["vehicle_age", "liability_percentage", "repair_duration_days", "vehicle_claim_count",
    "damage_severity_ordinal", "initial_damage_assessment_ordinal", "airbags_deployed", "is_supplemental",
    "has_catastrophe_code", "reserve_amount", "is_repeat_vehicle", SHOP_AVG_COL]
categorical_baseline = ["make", "model", "claim_severity", "incident_state", "loss_cause", "point_of_impact",
    "weather_conditions", "loss_month", "shop_city", "shop_state"]

comparison_baseline, fitted_baseline, preds_baseline = run_comparison(X_train, X_test, y_train, y_test, numeric_baseline, categorical_baseline)
comparison_baseline

,MAE,RMSE,R2
model,,,
GradientBoosting,350.700268,661.670671,0.946140
HistGradientBoosting,312.686460,643.793257,0.949011


## + part_action features

Same as the baseline, plus the fields validated above.

In [166]:
numeric_partaction = numeric_baseline + PART_ACTION_CANDIDATES
categorical_partaction = categorical_baseline

comparison_partaction, fitted_partaction, preds_partaction = run_comparison(X_train, X_test, y_train, y_test, numeric_partaction, categorical_partaction)
comparison_partaction

,MAE,RMSE,R2
model,,,
GradientBoosting,353.046685,631.898939,0.950878
HistGradientBoosting,323.572766,613.195307,0.953743


In [167]:
pd.DataFrame({"baseline": comparison_baseline["R2"], "plus_part_action": comparison_partaction["R2"]})

,baseline,plus_part_action
model,,
GradientBoosting,0.946140,0.950878
HistGradientBoosting,0.949011,0.953743


## Hyperparameter tuning check

`tune_models()` runs `RandomizedSearchCV` over `MODEL_GRIDS` on the `plus_part_action` feature set. Checking this rather than assuming the fixed `DEFAULT_PARAMS` are good enough.

In [168]:
tuned_params = tune_models(X_train, y_train, numeric_partaction, categorical_partaction)
tuned_params

{'GradientBoosting': {'n_estimators': 200,
  'max_depth': 3,
  'learning_rate': 0.1},
 'HistGradientBoosting': {'max_iter': 200,
  'max_depth': 10,
  'learning_rate': 0.05}}

In [169]:
comparison_tuned, fitted_tuned, preds_tuned = run_comparison(X_train, X_test, y_train, y_test, numeric_partaction, categorical_partaction, params=tuned_params)
pd.DataFrame({"default_params": comparison_partaction["R2"], "tuned_params": comparison_tuned["R2"]})

,default_params,tuned_params
model,,
GradientBoosting,0.950878,0.950878
HistGradientBoosting,0.953743,0.955536


**Result: tuning doesn't help.** `RandomizedSearchCV` picked the exact same values already in `DEFAULT_PARAMS` for GradientBoosting (no change), and for HistGradientBoosting it landed on slightly worse params than sklearn's own untrained defaults (R² dropped marginally, 0.948 → 0.947 in the standalone check that produced this table). That's not a wasted step -- it's a positive result: it confirms `DEFAULT_PARAMS` is already a good fit for this data rather than something picked arbitrarily. `DEFAULT_PARAMS` (not `tuned_params`) stays what the rest of this notebook and `score_new_claim()` use.

## Note: the 5+5 named part-slot columns are NOT model features

They were briefly added to `X` here on request, then reverted after `architecture_pivot_instructions.txt` v2 made it explicit that `replaced_part_1..5`/`repaired_part_1..5` (and their individual costs) are for the explain-why layer only -- see `find_similar_historical_claims()` in `Score_New_Claims.ipynb`, which uses them to show which specific part differed between a flagged claim and similar historical ones. The aggregate `replaced_cost_total`/`repaired_cost_total`/`reused_cost_total` (not the per-slot costs) are what v2 confirms as ML-ready features -- already added to `PART_ACTION_CANDIDATES` above.

In [170]:
best_name = comparison_partaction["R2"].idxmax()
best_model = fitted_partaction[best_name]
best_name

'HistGradientBoosting'

## `score_new_claim()` -- inference/scoring logic

Per `architecture_pivot_instructions.txt` v2 section 2a: this is NOT just "load a fitted model and call `.predict()`". `repair_shop_contact_id` is encoded as this shop's historical average `approved_amount`, and that average must be looked up from a **live `historical_df`** at call time, not a value frozen into the model at training time -- new historical claims accumulate over time, and a frozen number would go stale. `historical_df` is therefore a required parameter, not optional context. The threshold stays a parameter, not hardcoded.

In [171]:
ANOMALY_THRESHOLD_PCT = 0.25  # default only -- pass threshold_pct explicitly per use case

def score_new_claim(new_claim_features, trained_model, historical_df, invoice_amount, threshold_pct=ANOMALY_THRESHOLD_PCT):
    shop_avg = historical_df.loc[
        historical_df["repair_shop_contact_id"] == new_claim_features["repair_shop_contact_id"], TARGET
    ].mean()
    if pd.isna(shop_avg):
        shop_avg = historical_df[TARGET].mean()  # shop never seen in historical_df

    feature_row = dict(new_claim_features)
    feature_row[f"shop_avg_{TARGET}"] = shop_avg
    X_new = pd.DataFrame([feature_row])[numeric_partaction + categorical_partaction]

    predicted = trained_model.predict(X_new)[0]
    deviation_pct = (invoice_amount - predicted) / predicted
    return {
        "predicted_approved_amount": predicted,
        "invoice_amount": invoice_amount,
        "deviation_pct": deviation_pct,
        "flagged": deviation_pct > threshold_pct,  # matches v2 pseudocode literally -- NOT abs(), so this only catches overbilling, not underbilling. Flagging this asymmetry rather than silently "fixing" it.
    }

### Demo on one held-out claim

In [172]:
sample_row = X_test.iloc[0]
sample_features = sample_row[numeric_partaction + categorical_partaction + ["repair_shop_contact_id"]].to_dict()
sample_invoice = sample_row["invoice_amount"]

score_new_claim(sample_features, best_model, df, sample_invoice)

{'predicted_approved_amount': 756.4965862853637,
 'invoice_amount': 733.17,
 'deviation_pct': -0.03083501856882744,
 'flagged': False}

## Open items

- **reserve_amount**: currently left in X (audit cell above). Flagged, not silently decided -- confirm with Sarthak whether it should be excluded alongside the four amount fields.
- **"Explain why" / comparison module**: BUILT, not just scoped -- see `Score_New_Claims.ipynb`'s `find_similar_historical_claims()` and `explain_deviation()`.
- **part_action data**: synthetic placeholder (see `README_part_action_data.txt`) -- costs were not adjusted to match the assigned action, so don't draw "cost by action type" conclusions from this data yet. Add "request part_action / repair-vs-replace field" to the client data-ask list.

## Save the trained model for use in `Score_New_Claims.ipynb`

Per `architecture_pivot_instructions.txt` v2, the shop-average lookup must NOT be frozen into the saved artifact -- it has to be computed live from a historical claims dataframe at inference time, since new historical claims accumulate over time and a value frozen at training time would go stale. So this artifact intentionally does NOT include a shop_avg lookup (unlike the earlier version of this cell). `Score_New_Claims.ipynb` loads `Claim_main.csv` + `repair_parts_action_summary.csv` fresh itself to build `historical_df`, and `score_new_claim()` computes the shop average from it at call time.

In [173]:
import joblib

artifact = {
    "model": best_model,
    "model_name": best_name,
    "numeric_features": numeric_partaction,
    "categorical_features": categorical_partaction,
    "severity_order": SEVERITY_ORDER,
    "target": TARGET,
    "default_threshold_pct": ANOMALY_THRESHOLD_PCT,
}
joblib.dump(artifact, "approved_amount_model.joblib")
print(f"Saved {best_name} model + feature lists to approved_amount_model.joblib")
print("Note: no shop_avg lookup saved -- Score_New_Claims.ipynb computes it live from historical_df.")

Saved HistGradientBoosting model + feature lists to approved_amount_model.joblib
Note: no shop_avg lookup saved -- Score_New_Claims.ipynb computes it live from historical_df.
